# EquipED adapter -> GGUF LoRA conversion

Converts a trained PEFT LoRA adapter (from the DPO Colab training run) into a
GGUF LoRA file that llama.cpp's `llama-server` can load with `--lora`, on top
of the model file it already serves. The served model file is never modified.

Before running:

1. Any runtime works; no GPU is needed.
2. Upload the adapter zip to the **Files** panel (left sidebar) as
   `adapter.zip`. It is the zip stored on the backend under
   `adapters/<agent>/<adapter id>/adapter.zip`. There is no download button yet.
3. Runtime -> Run all.

Output: `adapter-f16.gguf` plus `adapter-f16.gguf.sha256`, downloaded at the
end. Give both to the host owner with `training/serving-lora-adapter.md`.

This only proves the file converts and is structurally a LoRA. Whether the
adapter actually helps is a separate evaluation.

In [ ]:
ADAPTER_ZIP = "adapter.zip"
# The adapter's own adapter_config.json names a 4-bit variant as its base
# (unsloth/gemma-3-4b-it-unsloth-bnb-4bit), which the converter cannot use.
# Point it at the regular instruct model, which is what the served Q4_0 file
# was quantized from.
BASE_MODEL_ID = "unsloth/gemma-3-4b-it"
OUTPUT_GGUF = "adapter-f16.gguf"
LLAMA_CPP_REPO = "https://github.com/ggml-org/llama.cpp"
# None = llama.cpp's default branch. To match the host's llama-server build,
# set a FULL commit hash from that build's source tree.
LLAMA_CPP_REF = None

In [ ]:
import hashlib
import json
import os
import subprocess
import sys
import zipfile
from pathlib import Path


def run(command):
    command = [str(part) for part in command]
    print("$", " ".join(command))
    subprocess.run(command, check=True)


def safe_extract(zip_path, target_dir):
    target = Path(target_dir).resolve()
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            destination = (target / member.filename).resolve()
            if destination != target and target not in destination.parents:
                raise ValueError(f"unsafe path in zip: {member.filename!r}")
        archive.extractall(target)


def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def check_lora_fields(
    general_type, adapter_type, alpha, tensors, expected_rank, expected_alpha
):
    problems = []
    if general_type != "adapter":
        problems.append(f"general.type is {general_type!r}, expected 'adapter'")
    if adapter_type != "lora":
        problems.append(f"adapter.type is {adapter_type!r}, expected 'lora'")
    if alpha is None or abs(float(alpha) - expected_alpha) > 1e-6:
        problems.append(f"alpha is {alpha!r}, expected {expected_alpha}")
    lora_a = [item for item in tensors if item[0].endswith(".lora_a")]
    lora_b = [item for item in tensors if item[0].endswith(".lora_b")]
    if not lora_a:
        problems.append("no lora_a tensors found")
    if len(lora_a) != len(lora_b):
        problems.append(f"{len(lora_a)} lora_a tensors but {len(lora_b)} lora_b tensors")
    for name, shape in lora_a + lora_b:
        if min(shape) != expected_rank:
            problems.append(
                f"{name}: smallest dimension {min(shape)} is not rank {expected_rank}"
            )
            break
    if problems:
        raise ValueError("; ".join(problems))
    return {"tensor_pairs": len(lora_a), "rank": expected_rank, "alpha": float(alpha)}


def verify_gguf_lora(path, expected_rank, expected_alpha):
    sys.path.insert(0, "llama.cpp/gguf-py")
    from gguf import GGUFReader

    reader = GGUFReader(path)

    def field_value(name):
        field = reader.fields.get(name)
        return None if field is None else field.contents()

    tensors = [
        (tensor.name, tuple(int(dim) for dim in tensor.shape))
        for tensor in reader.tensors
    ]
    return check_lora_fields(
        field_value("general.type"),
        field_value("adapter.type"),
        field_value("adapter.lora.alpha"),
        tensors,
        expected_rank,
        expected_alpha,
    )

In [ ]:
ADAPTER_DIR = Path("adapter_src")

if not os.path.exists(ADAPTER_ZIP):
    raise FileNotFoundError(
        f"Upload the adapter zip to the Files panel as {ADAPTER_ZIP!r} first."
    )
safe_extract(ADAPTER_ZIP, ADAPTER_DIR)

adapter_config = json.loads(
    (ADAPTER_DIR / "adapter_config.json").read_text(encoding="utf-8")
)
if adapter_config.get("peft_type") != "LORA":
    raise ValueError(f"not a LoRA adapter: peft_type={adapter_config.get('peft_type')!r}")
if not (ADAPTER_DIR / "adapter_model.safetensors").exists():
    raise FileNotFoundError("adapter_model.safetensors is missing from the zip")

LORA_RANK = int(adapter_config["r"])
LORA_ALPHA = float(adapter_config["lora_alpha"])
print("adapter base (as trained):", adapter_config.get("base_model_name_or_path"))
print("converter will use base config from:", BASE_MODEL_ID)
print("rank:", LORA_RANK, "alpha:", LORA_ALPHA)

In [ ]:
if not os.path.isdir("llama.cpp"):
    run(["git", "clone", "--depth", "1", LLAMA_CPP_REPO, "llama.cpp"])
    if LLAMA_CPP_REF:
        fetched = subprocess.run(
            ["git", "-C", "llama.cpp", "fetch", "--depth", "1", "origin", LLAMA_CPP_REF]
        )
        if fetched.returncode == 0:
            run(["git", "-C", "llama.cpp", "checkout", "FETCH_HEAD"])
        else:
            print(f"WARNING: could not fetch {LLAMA_CPP_REF}; using the default branch.")

requirements_dir = Path("llama.cpp/requirements")
candidates = [
    requirements_dir / "requirements-convert_lora_to_gguf.txt",
    requirements_dir / "requirements-convert_hf_to_gguf.txt",
]
requirements_file = next((path for path in candidates if path.exists()), None)
if requirements_file is None:
    raise FileNotFoundError(
        "no converter requirements file under llama.cpp/requirements; "
        "the llama.cpp layout may have changed, check that directory"
    )
run([sys.executable, "-m", "pip", "install", "-q", "-r", requirements_file])

In [ ]:
# Print the converter's own flags first, so the log records what this
# llama.cpp version actually accepts.
run([sys.executable, "llama.cpp/convert_lora_to_gguf.py", "--help"])

run(
    [
        sys.executable,
        "llama.cpp/convert_lora_to_gguf.py",
        "--base-model-id", BASE_MODEL_ID,
        "--outfile", OUTPUT_GGUF,
        "--outtype", "f16",
        ADAPTER_DIR,
    ]
)
if not os.path.exists(OUTPUT_GGUF):
    raise FileNotFoundError(f"converter finished but {OUTPUT_GGUF!r} was not created")

In [ ]:
summary = verify_gguf_lora(OUTPUT_GGUF, LORA_RANK, LORA_ALPHA)
print("GGUF LoRA verified:", summary)

In [ ]:
size_mb = os.path.getsize(OUTPUT_GGUF) / (1024 * 1024)
digest = sha256_of(OUTPUT_GGUF)
Path(OUTPUT_GGUF + ".sha256").write_text(f"{digest}  {OUTPUT_GGUF}\n", encoding="utf-8")
print(f"{OUTPUT_GGUF}: {size_mb:.1f} MB")
print("sha256:", digest)

try:
    from google.colab import files
except ImportError:
    print("Not running in Colab: the files are in the working directory.")
else:
    files.download(OUTPUT_GGUF)
    files.download(OUTPUT_GGUF + ".sha256")